# AniQuotes 4K: resumable Colab + Google Drive batch

This notebook is a separate draft-generation path. It does not call or replace the repository's local 9Router pipeline and it never publishes to Supabase or the app catalog.

- **Still batch:** 900 original anime-inspired SDXL images, rendered at 768 × 1368 and resized to 2160 × 3840. These are **upscaled 4K drafts**, not native 4K generations.
- **Live batch:** 80 muted 1080p, 24 fps, 10-second MP4 loops made from the first 80 completed stills with a subtle FFmpeg pan/zoom. This is 2D motion from one image, not frame-by-frame AI video.
- **Resume:** a stable job ID, atomic Drive manifest updates, deterministic seeds, SHA-256 duplicate checks, bounded retries and a lock file keep completed assets across Colab sessions.
- **Review:** every asset remains a draft with rights status `unreviewed`. Review appearance, similarity, and rights manually before any use.

Colab free GPU runtimes disconnect and have changing time limits. Run small session caps and reopen this notebook later to resume from the same Drive job. This is not a 24/7 worker.

Before generating, read Stability AI's SDXL license/model terms and confirm they fit your intended use. If Hugging Face asks for access, accept the model terms and store an access token in Colab **Secrets** as `HF_TOKEN`; never put it in a notebook cell. The token is read only for model download and is not written to Drive.


In [ ]:
# Install Python dependencies without changing Colab's preinstalled CUDA PyTorch.
import subprocess, sys
packages = [
    "diffusers>=0.32,<1.0", "transformers>=4.46,<5", "accelerate>=1.2",
    "safetensors>=0.4", "pillow>=10", "imageio-ffmpeg>=0.5",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dependencies installed. If Colab requests a runtime restart, restart and continue from the next setup cell.")


In [ ]:
# Mount Drive and confirm this is a GPU runtime before any model is loaded.
from google.colab import drive
from pathlib import Path
import torch

drive.mount("/content/drive")
if not torch.cuda.is_available():
    raise RuntimeError("In Colab, select Runtime > Change runtime type > T4 GPU, then rerun this cell.")
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GiB)")
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
print("HF_TOKEN secret available:", bool(HF_TOKEN))


In [ ]:
# Job settings. Keep JOB_ID and all generation settings the same to resume.
from pathlib import Path

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
NORMAL_TARGET = 900
LIVE_TARGET = 80  # first 80 stills, per this repository's live-wallpaper scope
BASE_WIDTH, BASE_HEIGHT = 768, 1368
OUTPUT_WIDTH, OUTPUT_HEIGHT = 2160, 3840
VIDEO_WIDTH, VIDEO_HEIGHT = 1080, 1920
VIDEO_FPS, VIDEO_SECONDS = 24, 10
INFERENCE_STEPS = 28
GUIDANCE_SCALE = 5.5

DRIVE_ROOT = Path("/content/drive/MyDrive/AniQuotes4K")
JOB_ROOT = DRIVE_ROOT / "wallpaper_jobs"
JOB_ID = "aniquotes-900-v1"  # retain this exact ID when resuming
JOB_DIR = JOB_ROOT / JOB_ID

# Per-session caps limit how long a free Colab runtime stays busy.
MAX_NORMAL_ASSETS_PER_SESSION = 25
MAX_LIVE_ASSETS_PER_SESSION = 10
RUN_NORMAL = False  # switch to True only when ready to generate stills
RUN_LIVE = False    # enable after stills are saved; video stage can resume separately

if LIVE_TARGET > NORMAL_TARGET:
    raise ValueError("LIVE_TARGET cannot exceed NORMAL_TARGET.")
print({
    "job": str(JOB_DIR), "stills": NORMAL_TARGET, "liveVideos": LIVE_TARGET,
    "stillSessionCap": MAX_NORMAL_ASSETS_PER_SESSION, "liveSessionCap": MAX_LIVE_ASSETS_PER_SESSION,
    "stillOutput": f"{OUTPUT_WIDTH}x{OUTPUT_HEIGHT} (upscaled draft)",
    "liveOutput": f"{VIDEO_WIDTH}x{VIDEO_HEIGHT} {VIDEO_FPS}fps x {VIDEO_SECONDS}s",
    "RUN_NORMAL": RUN_NORMAL, "RUN_LIVE": RUN_LIVE,
})


In [ ]:
import csv
import hashlib
import io
import json
import os
import re
import subprocess
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

from PIL import Image, ImageFilter

SUBJECTS = [
    "a lone apprentice astronomer", "a quiet shrine caretaker", "a courier with a red scarf", "a young botanist", "a masked stage performer",
    "a student carrying a sketchbook", "a tiny paper-crane maker", "a wandering tea seller", "a gentle forest guardian", "a rooftop violinist",
    "a traveler with a lantern", "a library apprentice", "a mountain climber", "a young mechanic", "a fox-like spirit made of light",
    "a night train conductor", "a baker before sunrise", "a kite maker", "a sea-glass collector", "a stargazing siblings duo",
    "a painter in a raincoat", "a bicycle messenger", "a sleepy cloud gardener", "a moon festival drummer", "a snowfield photographer",
    "a small robot learning to bloom", "a fisherman on a quiet pier", "a paper boat explorer", "a ceramic artist", "a traveler resting beneath a tree",
]
SETTINGS = [
    "a lantern-lit hillside town", "a floating garden above the clouds", "a misty cedar path", "a seaside railway platform", "an old bookshop at dusk",
    "a glass greenhouse in the rain", "a mountain lake under stars", "a quiet rooftop above the city", "a field of silver grass", "a warm train cabin",
    "a hidden waterfall grotto", "a narrow street after rainfall", "a moonlit bamboo grove", "a snowy village bridge", "a flower market at dawn",
    "a tiny observatory", "a cliffside lighthouse", "a peaceful harbor of paper boats", "a sunlit art classroom", "a festival avenue of wishes",
    "a desert garden under a blue moon", "a forest clearing with fireflies", "a coastal cliff in spring", "a tram stop in autumn", "a starlit orchard",
    "a hilltop windmill", "a quiet shrine courtyard", "a cloud-filled canyon", "a pastel city canal", "a meadow after a summer storm",
]
PALETTES = ["indigo and amber", "teal and peach", "lavender and silver", "emerald and gold", "coral and midnight blue", "soft jade and cream", "cobalt and rose", "plum and warm ivory", "icy blue and copper", "moss green and sunset orange", "rose pink and charcoal", "saffron and deep violet"]
LIGHTING = ["soft dawn rays", "a gentle crescent moon glow", "warm window light", "scattered firefly light", "a veil of luminous fog", "reflected city lights"]
COMPOSITIONS = ["a calm centered silhouette with generous open sky", "a low viewpoint with dramatic vertical depth", "a framed scene seen through branches", "a close foreground detail and distant layered scenery", "a graceful diagonal path leading toward the horizon", "a balanced asymmetrical composition with clear negative space"]
NEGATIVE_PROMPT = "text, letters, watermark, logo, signature, extra limbs, malformed hands, blurry, low detail, recognizable copyrighted character"
MAX_ATTEMPTS = 3


def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def make_prompt(index):
    # A mixed-radix traversal gives a broad, deterministic, non-repeating prompt set.
    total = len(SUBJECTS) * len(SETTINGS) * len(PALETTES) * len(LIGHTING) * len(COMPOSITIONS)
    key = (index * 7919 + 17371) % total
    subject = SUBJECTS[key % len(SUBJECTS)]
    key //= len(SUBJECTS)
    setting = SETTINGS[key % len(SETTINGS)]
    key //= len(SETTINGS)
    palette = PALETTES[key % len(PALETTES)]
    key //= len(PALETTES)
    lighting = LIGHTING[key % len(LIGHTING)]
    key //= len(LIGHTING)
    composition = COMPOSITIONS[key % len(COMPOSITIONS)]
    variation = str(index + 1).zfill(4)
    return (
        f"Original anime-inspired wallpaper study {variation}: {subject} in {setting}. "
        f"Use a distinctive {palette} palette, {lighting}, and {composition}. "
        "Refined hand-painted illustration, expressive but original character design, beautiful environmental detail, "
        "polished premium mobile wallpaper, portrait 9:16 framing, crisp edges, no text, no logos, no watermark, "
        "no recognizable existing franchise or copyrighted character."
    )


def current_config():
    return {
        "modelId": MODEL_ID,
        "normalTarget": NORMAL_TARGET,
        "liveTarget": LIVE_TARGET,
        "baseWidth": BASE_WIDTH,
        "baseHeight": BASE_HEIGHT,
        "outputWidth": OUTPUT_WIDTH,
        "outputHeight": OUTPUT_HEIGHT,
        "upscaleMethod": "Pillow Lanczos + UnsharpMask",
        "animationMethod": "FFmpeg seamless sinusoidal pan/zoom from a still image",
        "videoWidth": VIDEO_WIDTH,
        "videoHeight": VIDEO_HEIGHT,
        "videoFps": VIDEO_FPS,
        "videoSeconds": VIDEO_SECONDS,
        "inferenceSteps": INFERENCE_STEPS,
        "guidanceScale": GUIDANCE_SCALE,
        "promptVersion": "mixed-radix-v2",
    }


def atomic_write(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8", newline="") as stream:
            stream.write(text)
            stream.flush()
        os.replace(temp_name, path)
    except Exception:
        try:
            os.unlink(temp_name)
        except OSError:
            pass
        raise


def build_manifest():
    normal_items = []
    for index in range(1, NORMAL_TARGET + 1):
        prompt = make_prompt(index - 1)
        normal_items.append({
            "assetId": f"normal-{index:04d}",
            "index": index,
            "prompt": prompt,
            "promptSha256": sha256_bytes(prompt.encode("utf-8")),
            "status": "planned",
            "attempts": 0,
            "seed": None,
            "outputPath": None,
            "sha256": None,
            "width": None,
            "height": None,
            "native4k": False,
            "upscaleMethod": current_config()["upscaleMethod"],
            "rightsStatus": "unreviewed",
            "lastError": None,
        })
    live_items = []
    for index in range(1, LIVE_TARGET + 1):
        live_items.append({
            "assetId": f"live-{index:04d}",
            "index": index,
            "sourceAssetId": f"normal-{index:04d}",
            "status": "planned",
            "attempts": 0,
            "outputPath": None,
            "sha256": None,
            "width": VIDEO_WIDTH,
            "height": VIDEO_HEIGHT,
            "fps": VIDEO_FPS,
            "durationSeconds": VIDEO_SECONDS,
            "animationMethod": current_config()["animationMethod"],
            "rightsStatus": "unreviewed",
            "lastError": None,
        })
    return {
        "schemaVersion": 1,
        "jobId": JOB_ID,
        "createdAt": utc_now(),
        "updatedAt": utc_now(),
        "config": current_config(),
        "state": {"normal": "planned", "live": "planned"},
        "normal": normal_items,
        "live": live_items,
    }


def write_manifest(manifest):
    JOB_DIR.mkdir(parents=True, exist_ok=True)
    manifest["updatedAt"] = utc_now()
    atomic_write(JOB_DIR / "manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2) + "\n")
    rows = []
    for stage in ("normal", "live"):
        for item in manifest[stage]:
            rows.append({
                "job_id": manifest["jobId"],
                "stage": stage,
                "asset_id": item.get("assetId"),
                "source_asset_id": item.get("sourceAssetId", ""),
                "status": item.get("status"),
                "attempts": item.get("attempts", 0),
                "prompt_sha256": item.get("promptSha256", ""),
                "sha256": item.get("sha256") or "",
                "width": item.get("width") or "",
                "height": item.get("height") or "",
                "native_4k": item.get("native4k", ""),
                "quality_label": "upscaled_4k" if stage == "normal" else "1080p_loop",
                "output_path": item.get("outputPath") or "",
                "rights_status": item.get("rightsStatus", "unreviewed"),
                "last_error": (item.get("lastError") or "").replace("\n", " ")[:500],
            })
    csv_stream = io.StringIO(newline="")
    fields = list(rows[0].keys()) if rows else []
    writer = csv.DictWriter(csv_stream, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)
    atomic_write(JOB_DIR / "manifest.csv", csv_stream.getvalue())


def load_manifest():
    JOB_DIR.mkdir(parents=True, exist_ok=True)
    path = JOB_DIR / "manifest.json"
    if not path.exists():
        manifest = build_manifest()
        write_manifest(manifest)
        return manifest
    manifest = json.loads(path.read_text(encoding="utf-8"))
    if manifest.get("jobId") != JOB_ID or manifest.get("config") != current_config():
        raise RuntimeError("Saved job settings differ from this notebook. Keep the same configuration to resume, or choose a new JOB_ID.")
    if len(manifest.get("normal", [])) != NORMAL_TARGET or len(manifest.get("live", [])) != LIVE_TARGET:
        raise RuntimeError("Manifest item counts do not match the configured targets; it was preserved and generation stopped.")
    return manifest


def update_job_state(manifest):
    normal = manifest["normal"]
    live = manifest["live"]
    normal_done = sum(item["status"] == "completed" for item in normal)
    normal_failed = sum(item["status"] == "failed" for item in normal)
    normal_state = "complete" if normal_done == NORMAL_TARGET else ("attention_required" if normal_failed else "paused_or_incomplete")
    live_done = sum(item["status"] == "completed" for item in live)
    live_failed = sum(item["status"] == "failed" for item in live)
    live_state = "not_started" if all(item["status"] == "planned" for item in live) else ("complete" if live_done == LIVE_TARGET else ("attention_required" if live_failed else "paused_or_incomplete"))
    manifest["state"] = {"normal": normal_state, "live": live_state}


def acquire_job_lock():
    JOB_DIR.mkdir(parents=True, exist_ok=True)
    lock_path = JOB_DIR / "job.lock"
    try:
        fd = os.open(lock_path, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    except FileExistsError as exc:
        raise RuntimeError("A run already holds this job lock. If Colab disconnected, confirm no other runtime is active, then call clear_stale_lock().") from exc
    with os.fdopen(fd, "w", encoding="utf-8") as lock:
        json.dump({"pid": os.getpid(), "jobId": JOB_ID, "createdAt": utc_now()}, lock)
    return lock_path


def clear_stale_lock():
    lock_path = JOB_DIR / "job.lock"
    if lock_path.exists():
        print("Removing job lock. Only do this after confirming that no other Colab runtime is generating this job.")
        lock_path.unlink()
    else:
        print("No job lock exists.")


def existing_outputs_and_prompts():
    content_hashes = set()
    prompt_hashes = set()
    if not JOB_ROOT.exists():
        return content_hashes, prompt_hashes
    for manifest_path in JOB_ROOT.glob("*/manifest.json"):
        if manifest_path.parent.name == JOB_ID:
            continue
        try:
            previous = json.loads(manifest_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        for stage in ("normal", "live"):
            for item in previous.get(stage, []):
                if item.get("status") not in {"completed", "needs_review"}:
                    continue
                output = item.get("outputPath")
                if output and (manifest_path.parent / output).is_file():
                    if item.get("sha256"):
                        content_hashes.add(item["sha256"])
                    if item.get("promptSha256"):
                        prompt_hashes.add(item["promptSha256"])
    return content_hashes, prompt_hashes


def inspect_saved_image(path):
    path = Path(path)
    with Image.open(path) as image:
        image.verify()
    with Image.open(path) as image:
        width, height = image.size
        image.load()
    return {
        "sha256": sha256_file(path),
        "width": width,
        "height": height,
        "validImage": True,
        "exactOutputSize": (width, height) == (OUTPUT_WIDTH, OUTPUT_HEIGHT),
    }


def valid_saved_mp4(path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size < 32:
        return False
    with path.open("rb") as stream:
        header = stream.read(16)
    return b"ftyp" in header


def normal_path(item):
    return JOB_DIR / "normal" / f"{item['assetId']}.jpg"


def live_path(item):
    return JOB_DIR / "live" / f"{item['assetId']}.mp4"


def load_sdxl():
    import torch
    from diffusers import StableDiffusionXLPipeline

    if not torch.cuda.is_available():
        raise RuntimeError("Select a GPU runtime before loading SDXL.")
    pipe = StableDiffusionXLPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
    )
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_slicing()
    pipe.enable_vae_tiling()
    pipe.set_progress_bar_config(disable=True)
    return pipe


def deterministic_seed(asset_id):
    raw = f"{JOB_ID}:{asset_id}".encode("utf-8")
    return int.from_bytes(hashlib.sha256(raw).digest()[:4], "big") & 0x7FFFFFFF


def render_normal(pipe, item):
    import torch

    image = pipe(
        prompt=item["prompt"],
        negative_prompt=NEGATIVE_PROMPT,
        width=BASE_WIDTH,
        height=BASE_HEIGHT,
        num_inference_steps=INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=torch.Generator(device="cuda").manual_seed(item["seed"]),
    ).images[0]
    image = image.convert("RGB").resize((OUTPUT_WIDTH, OUTPUT_HEIGHT), Image.Resampling.LANCZOS)
    image = image.filter(ImageFilter.UnsharpMask(radius=1.0, percent=80, threshold=3))
    final_path = normal_path(item)
    final_path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = final_path.with_name(f".{final_path.stem}.{os.getpid()}.tmp.jpg")
    try:
        image.save(temp_path, format="JPEG", quality=94, optimize=True, progressive=True)
        inspection = inspect_saved_image(temp_path)
        if not inspection["exactOutputSize"]:
            raise RuntimeError("Saved still did not match the configured 2160x3840 output size.")
        return temp_path, inspection
    except Exception:
        temp_path.unlink(missing_ok=True)
        raise


def run_normal_batch(pipe, session_limit):
    if session_limit <= 0:
        raise ValueError("MAX_NORMAL_ASSETS_PER_SESSION must be positive.")
    lock_path = acquire_job_lock()
    try:
        manifest = load_manifest()
        content_hashes, prompt_hashes = existing_outputs_and_prompts()
        for item in manifest["normal"]:
            final_path = normal_path(item)
            if final_path.exists():
                try:
                    inspection = inspect_saved_image(final_path)
                    item.update({"sha256": inspection["sha256"], "width": inspection["width"], "height": inspection["height"], "outputPath": str(final_path.relative_to(JOB_DIR)).replace("\\", "/")})
                    if not inspection["exactOutputSize"]:
                        item["status"] = "needs_review"
                        item["lastError"] = "Existing file failed the configured output-size check."
                    elif item["sha256"] in content_hashes:
                        item["status"] = "duplicate"
                        item["lastError"] = "Existing file duplicates an earlier Drive asset."
                    else:
                        item["status"] = "completed"
                        item["lastError"] = None
                        content_hashes.add(item["sha256"])
                except Exception as exc:
                    item["status"] = "failed"
                    item["lastError"] = f"Existing image could not be verified: {type(exc).__name__}: {exc}"[:500]
                write_manifest(manifest)
            elif item["status"] == "running":
                item["status"] = "failed"
                item["lastError"] = "Colab stopped while this item was marked running; safe to retry from its deterministic seed."
                write_manifest(manifest)

        completed_this_session = 0
        for item in manifest["normal"]:
            if completed_this_session >= session_limit:
                break
            if item["status"] in {"completed", "needs_review", "duplicate"}:
                continue
            if item["promptSha256"] in prompt_hashes:
                item["status"] = "duplicate"
                item["lastError"] = "Prompt already appears in another Drive job; skipped to avoid duplicate work."
                write_manifest(manifest)
                continue
            if item["attempts"] >= MAX_ATTEMPTS:
                item["status"] = "failed"
                item["lastError"] = item.get("lastError") or f"Retry limit reached ({MAX_ATTEMPTS}). Use retry_failed('normal') after fixing the cause."
                continue
            completed_this_session += 1
            while item["attempts"] < MAX_ATTEMPTS:
                item["status"] = "running"
                item["attempts"] += 1
                item["seed"] = deterministic_seed(item["assetId"])
                item["lastError"] = None
                write_manifest(manifest)
                try:
                    temp_path, inspection = render_normal(pipe, item)
                    if inspection["sha256"] in content_hashes:
                        temp_path.unlink(missing_ok=True)
                        item.update({"status": "duplicate", "sha256": inspection["sha256"], "lastError": "Generated bytes match an existing Drive asset; duplicate was discarded."})
                    else:
                        os.replace(temp_path, normal_path(item))
                        content_hashes.add(inspection["sha256"])
                        item.update({
                            "status": "completed",
                            "sha256": inspection["sha256"],
                            "width": inspection["width"],
                            "height": inspection["height"],
                            "outputPath": str(normal_path(item).relative_to(JOB_DIR)).replace("\\", "/"),
                            "native4k": False,
                            "upscaleMethod": current_config()["upscaleMethod"],
                            "rightsStatus": "unreviewed",
                            "lastError": None,
                        })
                    write_manifest(manifest)
                    break
                except Exception as exc:
                    import torch
                    if "out of memory" in str(exc).lower() and torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    item["status"] = "failed"
                    item["lastError"] = f"{type(exc).__name__}: {exc}"[:500]
                    write_manifest(manifest)
                    if item["attempts"] >= MAX_ATTEMPTS:
                        break
                    time.sleep(min(5 * item["attempts"], 15))
        update_job_state(manifest)
        write_manifest(manifest)
        print_job_summary(manifest)
    finally:
        lock_path.unlink(missing_ok=True)


def retry_failed(stage):
    if stage not in {"normal", "live"}:
        raise ValueError("stage must be 'normal' or 'live'.")
    manifest = load_manifest()
    reset = 0
    for item in manifest[stage]:
        if item["status"] == "failed":
            item["previousError"] = item.get("lastError")
            item["status"] = "planned"
            item["attempts"] = 0
            item["lastError"] = None
            reset += 1
    write_manifest(manifest)
    print(f"Reset {reset} failed {stage} item(s) for retry. Their previous error is preserved in the manifest.")


def ffmpeg_path():
    import imageio_ffmpeg
    return imageio_ffmpeg.get_ffmpeg_exe()


def render_live(source_path, output_path):
    source_path = Path(source_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = output_path.with_name(f".{output_path.stem}.{os.getpid()}.tmp.mp4")
    ffmpeg = ffmpeg_path()
    cycle = VIDEO_FPS * VIDEO_SECONDS
    filter_graph = (
        f"zoompan=z='1+0.02*sin(2*PI*on/{cycle})':"
        f"x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d=1:"
        f"s={VIDEO_WIDTH}x{VIDEO_HEIGHT}:fps={VIDEO_FPS},format=yuv420p"
    )
    command = [
        ffmpeg, "-hide_banner", "-loglevel", "error", "-y",
        "-loop", "1", "-framerate", str(VIDEO_FPS), "-i", str(source_path),
        "-vf", filter_graph, "-t", str(VIDEO_SECONDS), "-r", str(VIDEO_FPS),
        "-an", "-c:v", "libx264", "-crf", "21", "-preset", "veryfast",
        "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(temp_path),
    ]
    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=max(300, VIDEO_SECONDS * 60))
        if result.returncode != 0:
            raise RuntimeError((result.stderr or "FFmpeg failed without a diagnostic.")[-500:])
        if not valid_saved_mp4(temp_path):
            raise RuntimeError("FFmpeg output is not a valid MP4 container.")
        return temp_path
    except Exception:
        temp_path.unlink(missing_ok=True)
        raise


def run_live_batch(session_limit):
    if session_limit <= 0:
        raise ValueError("MAX_LIVE_ASSETS_PER_SESSION must be positive.")
    lock_path = acquire_job_lock()
    try:
        manifest = load_manifest()
        live_by_id = {item["assetId"]: item for item in manifest["live"]}
        normal_by_id = {item["assetId"]: item for item in manifest["normal"]}
        other_hashes, _ = existing_outputs_and_prompts()
        completed_this_session = 0
        for item in manifest["live"]:
            final_path = live_path(item)
            if final_path.exists():
                if not valid_saved_mp4(final_path):
                    item["status"] = "needs_review"
                    item["lastError"] = "Existing video failed the MP4 container check."
                else:
                    digest = sha256_file(final_path)
                    item["sha256"] = digest
                    item["outputPath"] = str(final_path.relative_to(JOB_DIR)).replace("\\", "/")
                    if digest in other_hashes:
                        item["status"] = "duplicate"
                        item["lastError"] = "Existing video duplicates an earlier Drive asset."
                    else:
                        item["status"] = "completed"
                        item["lastError"] = None
                        other_hashes.add(digest)
                write_manifest(manifest)
            source = normal_by_id.get(item["sourceAssetId"])
            if not source or source.get("status") != "completed" or not normal_path(source).is_file():
                if item["status"] in {"planned", "blocked_source"}:
                    item["status"] = "blocked_source"
                    item["lastError"] = f"Waiting for {item['sourceAssetId']} to complete the still-image stage."
                    write_manifest(manifest)
                continue
            if item["status"] == "blocked_source":
                item["status"] = "planned"
                item["lastError"] = None
            if item["status"] in {"completed", "needs_review", "duplicate"}:
                continue
            if completed_this_session >= session_limit:
                break
            if item["attempts"] >= MAX_ATTEMPTS:
                item["status"] = "failed"
                item["lastError"] = item.get("lastError") or f"Retry limit reached ({MAX_ATTEMPTS}). Use retry_failed('live') after fixing the cause."
                continue
            completed_this_session += 1
            while item["attempts"] < MAX_ATTEMPTS:
                item["status"] = "running"
                item["attempts"] += 1
                item["lastError"] = None
                write_manifest(manifest)
                try:
                    temp_path = render_live(normal_path(source), final_path)
                    digest = sha256_file(temp_path)
                    if digest in other_hashes:
                        temp_path.unlink(missing_ok=True)
                        item.update({"status": "duplicate", "sha256": digest, "lastError": "Generated video matched an existing Drive asset and was discarded."})
                    else:
                        os.replace(temp_path, final_path)
                        other_hashes.add(digest)
                        item.update({"status": "completed", "sha256": digest, "outputPath": str(final_path.relative_to(JOB_DIR)).replace("\\", "/"), "width": VIDEO_WIDTH, "height": VIDEO_HEIGHT, "fps": VIDEO_FPS, "durationSeconds": VIDEO_SECONDS, "rightsStatus": "unreviewed", "lastError": None})
                    write_manifest(manifest)
                    break
                except Exception as exc:
                    item["status"] = "failed"
                    item["lastError"] = f"{type(exc).__name__}: {exc}"[:500]
                    write_manifest(manifest)
                    if item["attempts"] >= MAX_ATTEMPTS:
                        break
                    time.sleep(min(5 * item["attempts"], 15))
        update_job_state(manifest)
        write_manifest(manifest)
        print_job_summary(manifest)
    finally:
        lock_path.unlink(missing_ok=True)


def print_job_summary(manifest=None):
    manifest = manifest or load_manifest()
    for stage in ("normal", "live"):
        counts = {}
        for item in manifest[stage]:
            counts[item["status"]] = counts.get(item["status"], 0) + 1
        print(f"{stage}: {counts}")
    print(f"Manifest: {JOB_DIR / 'manifest.json'}")
    print(f"CSV:      {JOB_DIR / 'manifest.csv'}")


In [ ]:
# Plan/preflight only: verify 900 distinct prompts and write a Drive manifest without generating media.
prompts = [make_prompt(index) for index in range(NORMAL_TARGET)]
if len(set(prompts)) != NORMAL_TARGET:
    raise RuntimeError("Prompt catalog contains duplicates; generation stopped.")
manifest = load_manifest()
print(f"Unique prompts: {len(set(prompts))}/{NORMAL_TARGET}")
print_job_summary(manifest)
print("No image or video generation ran in this preflight cell.")


In [ ]:
# Still-image generation. Set RUN_NORMAL=True above, then run this cell.
if RUN_NORMAL:
    pipe = load_sdxl()
    try:
        run_normal_batch(pipe, MAX_NORMAL_ASSETS_PER_SESSION)
    finally:
        del pipe
        torch.cuda.empty_cache()
else:
    print("Still generation is disabled. Set RUN_NORMAL=True in the settings cell to start/resume this stage.")


In [ ]:
# Live MP4 generation. Set RUN_LIVE=True above after the still stage has saved source images.
if RUN_LIVE:
    run_live_batch(MAX_LIVE_ASSETS_PER_SESSION)
else:
    print("Live generation is disabled. Set RUN_LIVE=True in the settings cell to start/resume this stage.")


In [ ]:
# Inspect progress at any time. Statuses include planned, running, completed, needs_review,
# duplicate, blocked_source, and failed. Use retry_failed('normal') or retry_failed('live')
# after addressing a reported cause; the old error remains in previousError.
print_job_summary()
